# 강화학습 첫 경험 — Molecular Soup 버전

이 노트북은 방금 BIOTRON 시뮬레이션에서 한 것의 **아주 단순화된 버전**입니다. 목표:

1. 강화학습이 뭔지 직관적으로 이해
2. 환경(env) + 에이전트(agent) + 보상(reward)이 어떻게 맞물리는지 보기
3. PPO로 실제 학습시켜보기
4. **"리워드 해킹(reward hacking)"이라는 RL의 가장 유명한 함정 직접 체험**
5. BIOTRON에서 왜 에이전트가 `E=0`에 수렴했는지 이해

BIOTRON에서 우리가 발견한 것 — **에이전트가 "모든 것을 얼려버리는" 전략을 찾았고, 이건 리워드 설계의 허점 때문** — 이 예제에서도 똑같이 재현합니다.


## 1. 강화학습 30초 요약

```
   ┌──────────┐     action      ┌──────────────┐
   │          │ ───────────────>│              │
   │  Agent   │                 │ Environment  │
   │          │<─ obs, reward ──│              │
   └──────────┘                 └──────────────┘
```

- **Environment**: 우리가 만드는 "세상". 상태(state)를 가지고 있고, action을 받으면 상태가 변한다.
- **Agent**: 매 스텝 환경을 관찰(observation)하고 action을 선택한다.
- **Reward**: 환경이 agent에게 주는 "점수". agent의 목표는 이 점수의 총합을 최대화하는 것.
- **Policy**: agent의 "뇌" — 관찰을 받아서 어떤 action을 할지 결정하는 함수. 보통 신경망.
- **Training**: policy의 파라미터를 조정해서 더 많은 총 reward를 받도록 만드는 과정.

**PPO (Proximal Policy Optimization)**: OpenAI가 2017년에 만든 알고리즘. 현대 RL의 표준. 경험을 모아서 policy를 "너무 크게 바꾸지 않으면서" 조금씩 업데이트하는 방식.

## 2. 우리의 장난감 문제 — Molecular Soup

BIOTRON의 모든 복잡함을 빼고 핵심만 남긴 버전:

- **세상**: 수프 한 그릇. `monomers`(재료)와 `strands`(만들어진 것) 개수만 있다.
- **물리 법칙 (딱 2개)**:
  1. **형성(formation)**: 차가울수록(`T` 낮을수록) monomer가 strand로 빠르게 바뀐다.
  2. **분해(decay)**: 뜨거울수록(`T` 높을수록) strand가 monomer로 돌아간다.
- **에이전트의 액션**: 매 스텝 온도 `T ∈ [0, 1]`을 결정한다.
- **리워드**: 현재 strand 개수 (이게 함정!).
- **에피소드**: 100 스텝.

**질문**: 에이전트가 찾아낼 최적 `T`는 무엇일까?

상식적으로는 "중간 정도 (T≈0.3~0.5)" 같아요. 형성도 일어나고, 분해도 너무 심하지 않게. 근데 실제로 뭘 찾을지 볼까요?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO

np.random.seed(0)

## 3. 환경 만들기 — Gymnasium 인터페이스

강화학습 라이브러리들(stable-baselines3 포함)은 **Gymnasium**이라는 표준 인터페이스를 사용합니다. 환경 클래스 하나 만들면 됩니다:

- `reset()`: 새 에피소드 시작. 초기 obs 반환.
- `step(action)`: 한 스텝 진행. `(obs, reward, terminated, truncated, info)` 반환.
- `action_space`: 가능한 action의 범위
- `observation_space`: 가능한 obs의 범위

In [ ]:
class MolecularSoupEnv(gym.Env):
    """수프 한 그릇: monomer → strand 형성 + strand → monomer 분해.
    온도 T가 둘 다 컨트롤."""

    def __init__(self, episode_steps=100, reward_mode="accumulate"):
        super().__init__()
        # Action: 연속값, 내부에서 [0,1]로 rescale
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(1,), dtype=np.float32)
        # Observation: [strands, monomers, last_T] — 3차원
        self.observation_space = spaces.Box(low=0.0, high=10.0, shape=(3,), dtype=np.float32)
        self.episode_steps = episode_steps
        self.reward_mode = reward_mode  # "accumulate" 또는 "new_only"

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.monomers = 100
        self.strands = 0
        self.steps = 0
        self.last_T = 0.5
        return self._obs(), {}

    def step(self, action):
        # [-1, 1] → [0, 1]
        T = float((action[0] + 1) / 2)
        T = np.clip(T, 0.0, 1.0)

        # 물리 법칙 1: 형성 — 차가울수록 빠름
        form_rate = (1 - T) * 3.0
        formed = int(min(self.monomers, np.random.poisson(form_rate)))
        self.monomers -= formed
        self.strands += formed

        # 물리 법칙 2: 분해 — 뜨거울수록 빠름
        decay_p = T * 0.3
        decayed = int(np.random.binomial(self.strands, decay_p))
        self.strands -= decayed
        self.monomers += decayed

        # 리워드 — 이 부분이 나중에 바뀝니다!
        if self.reward_mode == "accumulate":
            reward = float(self.strands)  # 현재 strand 개수
        elif self.reward_mode == "new_only":
            reward = float(formed)  # 이번 스텝에 새로 만들어진 것만
        else:
            raise ValueError(self.reward_mode)

        self.steps += 1
        self.last_T = T
        truncated = self.steps >= self.episode_steps
        info = {"T": T, "formed": formed, "decayed": decayed,
                "strands": self.strands, "monomers": self.monomers}
        return self._obs(), reward, False, truncated, info

    def _obs(self):
        return np.array([
            self.strands / 100.0,
            self.monomers / 100.0,
            self.last_T,
        ], dtype=np.float32)

환경 테스트 — 초기 상태 확인:

In [ ]:
env = MolecularSoupEnv()
obs, _ = env.reset(seed=0)
print(f"초기 obs: {obs}  (strands={obs[0]*100:.0f}, monomers={obs[1]*100:.0f}, last_T={obs[2]:.2f})")
print(f"action space: {env.action_space}")
print(f"observation space: {env.observation_space}")

## 4. 리워드 landscape 탐색 (에이전트 없이)

RL 하기 전에 우리 손으로 문제를 이해하는 게 중요합니다. 여러 고정 온도 `T`에서 episode를 돌려서 총 reward가 어떻게 나오는지 봅시다.

In [ ]:
def run_constant_T(T_val, n_episodes=5, reward_mode="accumulate"):
    env = MolecularSoupEnv(reward_mode=reward_mode)
    rewards = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=ep)
        total = 0
        done = False
        action = np.array([2 * T_val - 1], dtype=np.float32)  # T → [-1,1]
        while not done:
            obs, r, term, trunc, _ = env.step(action)
            total += r
            done = term or trunc
        rewards.append(total)
    return np.mean(rewards), np.std(rewards)

Ts = np.linspace(0, 1, 21)
means = []
stds = []
for T in Ts:
    m, s = run_constant_T(T)
    means.append(m)
    stds.append(s)

plt.figure(figsize=(8, 4))
plt.errorbar(Ts, means, yerr=stds, marker='o')
plt.xlabel('Temperature T (constant over episode)')
plt.ylabel('Total episode reward (accumulate mode)')
plt.title('Reward landscape — reward = 현재 strand 개수')
plt.grid(alpha=0.3)
plt.show()
best_T = Ts[np.argmax(means)]
print(f"최고 T: {best_T:.2f}, mean reward: {max(means):.0f}")

**예상한 대로인가요?**

상식적으론 "중간쯤(T≈0.3~0.5)이 제일 좋을 것"이라고 생각했는데, 아마 `T=0` 근처가 최고로 나왔을 거예요. 왜냐하면:

- `T=0`: 형성은 최대(3.0 per step), 분해는 0 → strand가 계속 쌓이기만 함
- 매 스텝 reward = 현재 strand 개수 → strand가 많을수록 reward 큼
- `reward = strands`를 매 스텝 적분하면, strand가 영원히 많을 수록 좋음

**이게 BIOTRON에서 우리가 본 것과 정확히 같은 문제입니다.** 에이전트 없이도 이미 이 함정이 보여요.

## 5. PPO 에이전트 학습시키기

이제 진짜로 강화학습. stable-baselines3는 복잡한 내부 로직을 다 숨겨줘서 몇 줄이면 됩니다.

```python
model = PPO("MlpPolicy", env, ...)
model.learn(total_timesteps=...)
```

`"MlpPolicy"`는 "작은 다층 신경망 policy"를 의미. 우리 문제는 3차원 obs + 1차원 action이라 아주 작은 네트워크(기본값 64x64)면 충분.

In [ ]:
env_train = MolecularSoupEnv(reward_mode="accumulate")

model = PPO(
    "MlpPolicy",
    env_train,
    verbose=0,
    n_steps=256,     # 업데이트 전에 모으는 경험 개수
    batch_size=64,
    n_epochs=10,
    learning_rate=3e-4,
    ent_coef=0.02,   # 탐색(exploration) 장려 정도
    gamma=0.99,      # 미래 reward 할인율
    policy_kwargs=dict(net_arch=[32, 32]),  # 작은 네트워크
    device="cpu",
)

print("학습 시작...")
model.learn(total_timesteps=20_000, progress_bar=False)
print("학습 완료!")

## 6. 학습된 policy 평가

에이전트가 뭘 배웠나 봅시다. deterministic=True로 하면 policy의 mean action을 사용 (평가 때는 탐색 불필요).

In [ ]:
def run_agent(model, env, n_episodes=5):
    rewards = []
    T_history = []
    strand_history = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=100 + ep)
        total = 0
        Ts = []
        strands = []
        done = False
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, r, term, trunc, info = env.step(action)
            total += r
            Ts.append(info["T"])
            strands.append(info["strands"])
            done = term or trunc
        rewards.append(total)
        T_history.append(Ts)
        strand_history.append(strands)
    return rewards, T_history, strand_history

env_eval = MolecularSoupEnv(reward_mode="accumulate")
rewards, T_hist, strand_hist = run_agent(model, env_eval, n_episodes=5)
print(f"Agent mean reward: {np.mean(rewards):.0f} ± {np.std(rewards):.0f}")
print(f"Agent mean T during episode: {np.mean([np.mean(t) for t in T_hist]):.3f}")
print(f"For comparison: best constant T = {best_T:.2f} with reward {max(means):.0f}")

In [ ]:
# Agent의 T 선택을 시간에 따라 플롯
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for ep, Ts in enumerate(T_hist):
    ax1.plot(Ts, alpha=0.5, label=f'ep {ep}')
ax1.set_xlabel('step')
ax1.set_ylabel('T (chosen by agent)')
ax1.set_title('Agent의 temperature schedule')
ax1.set_ylim(-0.05, 1.05)
ax1.grid(alpha=0.3)

for ep, ss in enumerate(strand_hist):
    ax2.plot(ss, alpha=0.5, label=f'ep {ep}')
ax2.set_xlabel('step')
ax2.set_ylabel('strand count')
ax2.set_title('Strand 개수 (시간에 따른)')
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**관찰**: 에이전트가 `T=0` 근처에 고정된 걸 볼 수 있을 겁니다. 그리고 strand 개수는 계속 쌓여서 ~100에 도달(monomer를 다 쓸 때까지).

에이전트가 "똑똑해서" 이걸 찾은 게 아니라, **우리 리워드가 이 전략을 가장 잘 보상하도록 설계되었기 때문**입니다.

### 이게 바로 우리가 BIOTRON에서 본 현상

BIOTRON에서:
- Agent가 `E=0`에 수렴 → 우리가 "local optimum에 빠졌나?" 의심
- 사실은 **우리 reward 설계가 "frozen = best"를 만들었음**
- Agent는 정직하게 reward를 최대화한 것

RL 분야에서 이걸 **"reward hacking"** 또는 **"specification gaming"**이라고 부릅니다. 에이전트는 당신이 원하는 걸 배우는 게 아니라, 당신이 측정하는 걸 배웁니다.

> **"If you can't measure it, you can't optimize it. And if you optimize a bad measure, you get a bad outcome."**

## 7. 리워드를 고치면 — "new_only" 모드

고치는 방법: reward를 **"현재 strand 개수"** 가 아니라 **"이번 스텝에 새로 만들어진 strand 개수"** 로 바꿈.

이러면:
- `T=0`: 초기엔 빠르게 만들어지지만, monomer가 떨어지면 더 이상 형성 X → reward 0
- `T=중간`: 지속적으로 분해와 재형성이 일어남 → 매 스텝 조금씩 reward → 총 reward 높음
- `T=1`: 형성 0 → reward 0

즉 **"얼리는 전략"은 더 이상 최적이 아님**. 에이전트는 turnover가 있는 중간 온도를 찾아야 함.

In [ ]:
# 고친 reward로 landscape 다시 그리기
Ts = np.linspace(0, 1, 21)
means_new = []
for T in Ts:
    m, _ = run_constant_T(T, reward_mode="new_only")
    means_new.append(m)

plt.figure(figsize=(8, 4))
plt.plot(Ts, means, 'o-', label='reward = strands (old, broken)')
plt.plot(Ts, means_new, 's-', label='reward = formed (new, fixed)')
plt.xlabel('Temperature T')
plt.ylabel('Total episode reward')
plt.title('Reward landscape 비교')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

best_T_new = Ts[np.argmax(means_new)]
print(f"'new_only' 모드 최고 T: {best_T_new:.2f}, reward: {max(means_new):.1f}")

이제 `T=0`가 아닌 중간 어딘가가 최고일 거예요. 리워드를 바꾸는 것만으로 최적 전략이 질적으로 달라졌습니다.

이제 이 새 리워드로 에이전트를 다시 학습시켜 봅시다:

In [ ]:
env_train_new = MolecularSoupEnv(reward_mode="new_only")
model_new = PPO(
    "MlpPolicy",
    env_train_new,
    verbose=0,
    n_steps=256,
    batch_size=64,
    n_epochs=10,
    learning_rate=3e-4,
    ent_coef=0.02,
    gamma=0.99,
    policy_kwargs=dict(net_arch=[32, 32]),
    device="cpu",
)
print("새 리워드로 학습 시작...")
model_new.learn(total_timesteps=20_000, progress_bar=False)
print("완료!")

env_eval_new = MolecularSoupEnv(reward_mode="new_only")
rewards_new, T_hist_new, _ = run_agent(model_new, env_eval_new, n_episodes=5)
print(f"새 agent mean reward: {np.mean(rewards_new):.1f} ± {np.std(rewards_new):.1f}")
print(f"새 agent mean T: {np.mean([np.mean(t) for t in T_hist_new]):.3f}")

In [ ]:
# 새 agent의 T schedule
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
for ep, Ts in enumerate(T_hist_new):
    ax.plot(Ts, alpha=0.5, label=f'ep {ep}')
ax.set_xlabel('step')
ax.set_ylabel('T')
ax.set_title("'new_only' 리워드로 학습한 agent의 temperature schedule")
ax.set_ylim(-0.05, 1.05)
ax.axhline(best_T_new, color='red', linestyle='--', alpha=0.5, label=f'best constant ({best_T_new:.2f})')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

이제 에이전트가 질적으로 다른 전략을 찾았을 겁니다. 더 이상 `T=0`이 아니라 어느 중간값 근처. 이게 **리워드 재설계의 힘**입니다.

## 8. 큰 교훈

### RL의 철학

1. **에이전트는 당신의 진짜 의도를 모릅니다.** 에이전트는 오직 숫자(reward)만 보고 그걸 최대화합니다.
2. **측정되지 않는 것은 최적화되지 않습니다.** 당신이 원하는 property를 명시적으로 reward에 넣어야 합니다.
3. **Reward hacking은 버그가 아니라 feature입니다.** 에이전트가 의도하지 않은 전략을 찾는다면, 그건 에이전트가 못한 게 아니라 당신의 reward가 불완전하다는 신호입니다.
4. **항상 landscape를 먼저 봅시다.** RL 돌리기 전에 handcrafted policy로 reward landscape를 그려보면 많은 버그가 미리 보입니다.

### BIOTRON에서의 실제 예

BIOTRON에서 우리가 본 것:
- 에이전트가 `E=0` (최저 온도)에 수렴
- "좋은 발견!" 이라고 자랑할 수도 있었지만, 사실은 **reward 설계의 허점**
- `reward = n_compartments` 였는데, 일단 만든 compartment가 decay 없이 영원히 유지되니까 얼리는 게 최적
- 진짜 원했던 건 "계속 emergence가 일어나는 다이나믹한 시스템" — 이건 `reward = births per step × compartment_factor` 같은 걸로 측정해야 함

### 이 문제는 RL 분야 전체의 핵심 난제

- OpenAI의 CoastRunners 실험: 게임 에이전트가 점수 아이템을 무한 반복해서 경주를 안 함
- 로봇 팔 블록 쌓기: "위에 올린다"만 reward → 로봇이 자기 손으로 쥐고 올려서 reward 받음
- Alignment 문제 전체가 사실 reward specification 문제의 일반화

### 결론

> **"The agent is always right. The reward is always wrong."**

강화학습의 90%는 실제로 알고리즘 튜닝이 아니라 **"내가 진짜로 원하는 게 뭔지"를 수학적으로 명확히 표현하는 것**입니다. 이게 어렵고, 이게 재미있습니다.

## 9. 다음 단계 (원하시면)

- `n_steps`, `ent_coef`, `learning_rate`, 네트워크 크기 등 하이퍼파라미터 바꿔보기
- reward에 "diversity 보너스" 추가해보기 (여러 종류 strand 가정)
- action space를 2차원으로 확장 (e.g., 온도 + 재료 투입량)
- curriculum learning: 처음엔 쉬운 과제, 점차 어려운 과제

- **진짜 BIOTRON**: 위와 똑같은 구조지만, 환경이 훨씬 복잡하고, 물리가 emergent self-replication + compartmentalization을 만들어냄. 우리가 방금 한 작업.
